<a href="https://colab.research.google.com/github/tobiasllop/Tesis/blob/main/Tesis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tesis 2026 - Tobias Llop

In [ ]:
# import Libraries
import pandas as pd
import numpy as np
import matplotlib as plt

In [ ]:
import pandas as pd
import pyarrow.parquet as pq
from google.colab import drive

drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/Tesis2026/deudores_final_enero.parquet'
prefijos_fisicas = ('20', '23', '24', '27')

# Filtrado dinámico de TODO el dataset
parquet_file = pq.ParquetFile(file_path)
filtered_chunks = []
total_rows_scanned = 0

print("Iniciando el procesamiento del dataset completo...")
print("Esto puede tomar unos minutos dependiendo del tamaño del archivo.")

# Iteramos por todo el archivo sin límite de filas (target_rows)
for batch in parquet_file.iter_batches(batch_size=200000):
    chunk_df = batch.to_pandas()
    total_rows_scanned += len(chunk_df)

    # Asegurar tipos para el filtrado
    chunk_df['tipo_id'] = chunk_df['tipo_id'].astype(str)
    chunk_df['nro_id'] = chunk_df['nro_id'].astype(str)

    # Aplicar filtro de exclusión (Personas Jurídicas)
    is_fisica = (chunk_df['tipo_id'] == '11') & (chunk_df['nro_id'].str.startswith(prefijos_fisicas))
    juridicas_chunk = chunk_df[~is_fisica]

    if not juridicas_chunk.empty:
        filtered_chunks.append(juridicas_chunk)

    # Feedback visual del progreso
    if total_rows_scanned % 1000000 == 0:
        print(f"Filas procesadas: {total_rows_scanned}...")

# Combinar todos los resultados filtrados
if filtered_chunks:
    df = pd.concat(filtered_chunks)
    print(f"¡Procesamiento completado!")
    print(f"Total de filas analizadas: {total_rows_scanned}")
    print(f"Total de 'personas jurídicas' encontradas: {len(df)}")
    display(df.head())
else:
    print("No se encontraron 'personas jurídicas' en todo el dataset.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Iniciando el procesamiento del dataset completo...
Esto puede tomar unos minutos dependiendo del tamaño del archivo.
Filas procesadas: 1000000...
Filas procesadas: 2000000...
Filas procesadas: 3000000...
Filas procesadas: 4000000...
Filas procesadas: 5000000...
Filas procesadas: 6000000...
Filas procesadas: 7000000...
Filas procesadas: 8000000...
Filas procesadas: 9000000...
Filas procesadas: 10000000...
Filas procesadas: 11000000...
Filas procesadas: 12000000...
Filas procesadas: 13000000...
Filas procesadas: 14000000...
Filas procesadas: 15000000...
Filas procesadas: 16000000...
Filas procesadas: 17000000...
Filas procesadas: 18000000...
Filas procesadas: 19000000...
Filas procesadas: 20000000...
Filas procesadas: 21000000...
Filas procesadas: 22000000...
Filas procesadas: 23000000...
Filas procesadas: 24000000...
Filas procesadas: 25000000...
Filas procesa

,cod_entidad,fecha,tipo_id,nro_id,actividad,situacion,prestamos,sin_uso,garantias_otorg,otros_conceptos,...,contragar_pref_B,sin_contragar_pref,previsiones,deuda_cubierta,proceso_judicial,refinanciaciones,recat_obligatoria,sit_juridica,irrecuperable,dias_atraso
83468,00007,202601,11,30500057102,651.0,1,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
83540,00007,202601,11,30500492283,949.0,1,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
83612,00007,202601,11,30501674504,12.0,1,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
83684,00007,202601,11,30502112259,162.0,1,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
83756,00007,202601,11,30503610988,61.0,1,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0


In [ ]:
file_path_entidades = '/content/drive/MyDrive/Tesis2026/EntidadesFinancierasBCRA.csv'
df_entidades = pd.read_csv(file_path_entidades)

# Seleccionamos solo las columnas necesarias y nos aseguramos de que no haya duplicados
df_entidades_selected = df_entidades[['cod_entidad', 'nombre_entidad', 'grupo_entidad']].copy()

# Limpiamos cod_entidad
df['cod_entidad'] = df['cod_entidad'].astype(str).str.zfill(5)
df_entidades_selected['cod_entidad'] = df_entidades_selected['cod_entidad'].astype(str).str.zfill(5)

# Eliminamos columnas si ya existen en df para evitar sufijos _x _y
columns_to_add = ['nombre_entidad', 'grupo_entidad']
df = df.drop(columns=[c for c in columns_to_add if c in df.columns])

# Realizamos el merge
df_merged = pd.merge(df, df_entidades_selected, on='cod_entidad', how='left')

# Reordenamos las columnas
current_columns = df_merged.columns.tolist()
idx_cod_entidad = current_columns.index('cod_entidad')

for col in columns_to_add:
    if col in current_columns:
        current_columns.remove(col)

new_columns_order = current_columns[:idx_cod_entidad + 1] + columns_to_add + current_columns[idx_cod_entidad + 1:]
df = df_merged[new_columns_order]

print("DataFrame actualizado con nombres de entidades:")
display(df.head())
display(df.info())

DataFrame actualizado con nombres de entidades:


,cod_entidad,nombre_entidad,grupo_entidad,fecha,tipo_id,nro_id,actividad,situacion,prestamos,sin_uso,...,contragar_pref_B,sin_contragar_pref,previsiones,deuda_cubierta,proceso_judicial,refinanciaciones,recat_obligatoria,sit_juridica,irrecuperable,dias_atraso
0,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30500057102,651.0,1,NaN,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
1,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30500492283,949.0,1,NaN,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
2,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30501674504,12.0,1,NaN,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
3,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30502112259,162.0,1,NaN,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
4,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30503610988,61.0,1,NaN,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 473653 entries, 0 to 473652
Data columns (total 26 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   cod_entidad         473653 non-null  object 
 1   nombre_entidad      432428 non-null  object 
 2   grupo_entidad       432428 non-null  object 
 3   fecha               473653 non-null  object 
 4   tipo_id             473653 non-null  object 
 5   nro_id              473653 non-null  object 
 6   actividad           434673 non-null  float64
 7   situacion           473653 non-null  object 
 8   prestamos           0 non-null       float64
 9   sin_uso             0 non-null       float64
 10  garantias_otorg     0 non-null       float64
 11  otros_conceptos     0 non-null       float64
 12  garantia_pref_A     0 non-null       float64
 13  garantia_pref_B     0 non-null       float64
 14  sin_garantia_pref   0 non-null       float64
 15  contragar_pref_A    0 non-null    

None

In [ ]:
print(df.head())

  cod_entidad                          nombre_entidad grupo_entidad   fecha  \
0       00007  BANCO DE GALICIA Y BUENOS AIRES S.A.U.             A  202601   
1       00007  BANCO DE GALICIA Y BUENOS AIRES S.A.U.             A  202601   
2       00007  BANCO DE GALICIA Y BUENOS AIRES S.A.U.             A  202601   
3       00007  BANCO DE GALICIA Y BUENOS AIRES S.A.U.             A  202601   
4       00007  BANCO DE GALICIA Y BUENOS AIRES S.A.U.             A  202601   

  tipo_id       nro_id  actividad situacion  prestamos  sin_uso  ...  \
0      11  30500057102      651.0         1        NaN      NaN  ...   
1      11  30500492283      949.0         1        NaN      NaN  ...   
2      11  30501674504       12.0         1        NaN      NaN  ...   
3      11  30502112259      162.0         1        NaN      NaN  ...   
4      11  30503610988       61.0         1        NaN      NaN  ...   

   contragar_pref_B  sin_contragar_pref  previsiones  deuda_cubierta  \
0               NaN 

In [ ]:
output_path = '/content/drive/MyDrive/Tesis2026/'

# Prepare list to store top 10 dataframes for export
top_10_dfs = {}

for i in range(1, 6):
    # Filter dataframe by 'situacion'
    current_df = df[df['situacion'] == str(i)].copy()

    # Ensure 'prestamos' is numeric and fill NaNs with 0
    current_df['prestamos'] = pd.to_numeric(current_df['prestamos'], errors='coerce').fillna(0)

    # Sort by 'prestamos' in descending order and get the top 10
    df_situacion_top_10 = current_df.sort_values(by='prestamos', ascending=False).head(10)

    # Store the top 10 dataframe
    top_10_dfs[f'df_situacion_{i}'] = df_situacion_top_10

    print(f"\nTop 10 de df_situacion_{i} (ordenado por prestamos descendente):")
    display(df_situacion_top_10)

    # Export to Excel
    excel_filename = f'{output_path}df_situacion_{i}_top10.xlsx'
    df_situacion_top_10.to_excel(excel_filename, index=False)
    print(f"Exportado df_situacion_{i}_top10 a: {excel_filename}")

print("\n¡Proceso completado! Todos los dataframes top 10 han sido creados y exportados.")


Top 10 de df_situacion_1 (ordenado por prestamos descendente):


,cod_entidad,nombre_entidad,grupo_entidad,fecha,tipo_id,nro_id,actividad,situacion,prestamos,sin_uso,...,contragar_pref_B,sin_contragar_pref,previsiones,deuda_cubierta,proceso_judicial,refinanciaciones,recat_obligatoria,sit_juridica,irrecuperable,dias_atraso
473652,72634,NaN,NaN,202601,11,33717950289,NaN,1,0.0,NaN,...,NaN,NaN,0.0,0,0,9,0,9,9,0
0,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30500057102,651.0,1,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
1,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30500492283,949.0,1,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
2,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30501674504,12.0,1,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
3,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30502112259,162.0,1,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
4,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30503610988,61.0,1,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
5,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30503658093,162.0,1,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
6,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30503738739,271.0,1,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
7,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30503858327,465.0,1,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
8,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30504048078,271.0,1,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0


Exportado df_situacion_1_top10 a: /content/drive/MyDrive/Tesis2026/df_situacion_1_top10.xlsx

Top 10 de df_situacion_2 (ordenado por prestamos descendente):


,cod_entidad,nombre_entidad,grupo_entidad,fecha,tipo_id,nro_id,actividad,situacion,prestamos,sin_uso,...,contragar_pref_B,sin_contragar_pref,previsiones,deuda_cubierta,proceso_judicial,refinanciaciones,recat_obligatoria,sit_juridica,irrecuperable,dias_atraso
473624,72634,NaN,NaN,202601,11,30717867013,NaN,2,0.0,NaN,...,NaN,NaN,0.0,0,0,9,0,9,9,0
388,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30500597557,275.0,2,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
695,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30500487379,829.0,2,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
875,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30501099933,201.0,2,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,1,0,2384
1364,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30598702388,949.0,2,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
1403,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30591912654,524.0,2,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
1457,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30534958680,469.0,2,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
1564,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30536703396,461.0,2,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
1719,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30619401723,14.0,2,0.0,NaN,...,NaN,NaN,0.0,0,0,1,0,0,0,0
1929,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30595066588,682.0,2,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0


Exportado df_situacion_2_top10 a: /content/drive/MyDrive/Tesis2026/df_situacion_2_top10.xlsx

Top 10 de df_situacion_3 (ordenado por prestamos descendente):


...

Exportado df_situacion_3_top10 a: /content/drive/MyDrive/Tesis2026/df_situacion_3_top10.xlsx

Top 10 de df_situacion_4 (ordenado por prestamos descendente):


,cod_entidad,nombre_entidad,grupo_entidad,fecha,tipo_id,nro_id,actividad,situacion,prestamos,sin_uso,...,contragar_pref_B,sin_contragar_pref,previsiones,deuda_cubierta,proceso_judicial,refinanciaciones,recat_obligatoria,sit_juridica,irrecuperable,dias_atraso
473637,72634,NaN,NaN,202601,11,30718652886,NaN,4,0.0,NaN,...,NaN,NaN,0.0,0,0,9,0,9,9,0
437,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30588353466,949.0,4,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
687,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30563046682,682.0,4,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
696,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30563078347,102.0,4,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
472824,72634,NaN,NaN,202601,11,30718257952,NaN,4,0.0,NaN,...,NaN,NaN,0.0,0,0,9,0,9,9,0
472850,72634,NaN,NaN,202601,11,30717350436,NaN,4,0.0,NaN,...,NaN,NaN,0.0,0,0,9,1,9,9,0
472861,72634,NaN,NaN,202601,11,30718013387,NaN,4,0.0,NaN,...,NaN,NaN,0.0,0,0,9,0,9,9,0
472870,72634,NaN,NaN,202601,11,30718177843,NaN,4,0.0,NaN,...,NaN,NaN,0.0,0,0,9,0,9,9,0
472904,72634,NaN,NaN,202601,11,30718373456,NaN,4,0.0,NaN,...,NaN,NaN,0.0,0,0,9,1,9,9,53
472948,72634,NaN,NaN,202601,11,30715754637,NaN,4,0.0,NaN,...,NaN,NaN,0.0,0,0,9,0,9,9,0


Exportado df_situacion_4_top10 a: /content/drive/MyDrive/Tesis2026/df_situacion_4_top10.xlsx

Top 10 de df_situacion_5 (ordenado por prestamos descendente):


,cod_entidad,nombre_entidad,grupo_entidad,fecha,tipo_id,nro_id,actividad,situacion,prestamos,sin_uso,...,contragar_pref_B,sin_contragar_pref,previsiones,deuda_cubierta,proceso_judicial,refinanciaciones,recat_obligatoria,sit_juridica,irrecuperable,dias_atraso
473609,72634,NaN,NaN,202601,11,30716673223,NaN,5,0.0,NaN,...,NaN,NaN,0.0,0,0,9,0,9,9,0
219,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30521822194,452.0,5,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
223,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30525825902,931.0,5,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
269,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30535190050,292.0,5,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,1,0,757
284,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30536122784,682.0,5,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
357,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30525966859,475.0,5,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,1,0,1705
684,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30532402391,682.0,5,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
887,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30511715829,14.0,5,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
1166,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30547065650,682.0,5,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0
1218,00007,BANCO DE GALICIA Y BUENOS AIRES S.A.U.,A,202601,11,30580746248,682.0,5,0.0,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0


Exportado df_situacion_5_top10 a: /content/drive/MyDrive/Tesis2026/df_situacion_5_top10.xlsx

¡Proceso completado! Todos los dataframes top 10 han sido creados y exportados.


### Test: Cargar y filtrar por `nro_id` específico desde el archivo original

In [ ]:
import pandas as pd
import pyarrow.parquet as pq

# La ruta al archivo ya está definida como file_path en celdas anteriores

# Definimos el nro_id que queremos buscar
target_nro_id = '20455010366'

# Lista para almacenar las filas encontradas
found_rows = []

print(f"Iniciando búsqueda de nro_id: {target_nro_id} en el archivo original: {file_path}...")

# Abrimos el archivo Parquet para iterar sobre él
parquet_file_reader = pq.ParquetFile(file_path)

# Iteramos sobre los lotes del archivo para optimizar el uso de memoria
# Podrías ajustar `batch_size` si tienes problemas de memoria, pero 200000 suele ser un buen tamaño
for batch in parquet_file_reader.iter_batches(batch_size=200000):
    current_chunk_df = batch.to_pandas()

    # Aseguramos que la columna 'nro_id' sea de tipo string para la comparación
    current_chunk_df['nro_id'] = current_chunk_df['nro_id'].astype(str)

    # Filtramos el chunk actual por el nro_id objetivo
    matched_row_in_chunk = current_chunk_df[current_chunk_df['nro_id'] == target_nro_id]

    # Si encontramos la fila, la añadimos a nuestra lista y podemos parar (asumiendo nro_id único)
    if not matched_row_in_chunk.empty:
        found_rows.append(matched_row_in_chunk)
        # Si solo esperas una fila por nro_id, puedes descomentar la siguiente línea para detener la búsqueda
        # break

# Si se encontró alguna fila, las concatenamos en un solo DataFrame y las mostramos
if found_rows:
    df_test_cuil = pd.concat(found_rows, ignore_index=True)
    print(f"¡Fila(s) con nro_id {target_nro_id} encontrada(s) en el archivo original!")
    display(df_test_cuil)
else:
    print(f"No se encontró ninguna fila con nro_id: {target_nro_id} en el archivo original.")

Iniciando búsqueda de nro_id: 20455010366 en el archivo original: /content/drive/MyDrive/Tesis2026/deudores_final_enero.parquet...
¡Fila(s) con nro_id 20455010366 encontrada(s) en el archivo original!


,cod_entidad,fecha,tipo_id,nro_id,actividad,situacion,prestamos,sin_uso,garantias_otorg,otros_conceptos,...,contragar_pref_B,sin_contragar_pref,previsiones,deuda_cubierta,proceso_judicial,refinanciaciones,recat_obligatoria,sit_juridica,irrecuperable,dias_atraso
0,00015,202601,11,20455010366,0.0,1,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,0,0,0,0,0,0,0


In [ ]:
print(f"Número total de filas en el DataFrame: {len(df)}")
print(f"Número de valores no nulos en la columna 'prestamos': {df['prestamos'].count()}")

# Display rows where 'prestamos' is not NaN
non_nan_prestamos = df[df['prestamos'].notna()]
if not non_nan_prestamos.empty:
    print("\nPrimeras 5 filas donde 'prestamos' NO es NaN:")
    display(non_nan_prestamos.head())
else:
    print("\nTodos los valores en la columna 'prestamos' son NaN.")

Número total de filas en el DataFrame: 473653
Número de valores no nulos en la columna 'prestamos': 0

Todos los valores en la columna 'prestamos' son NaN.
